# 01 — Data Cleaning

**Trinity University Football Analytics**

This notebook loads all raw Hudl game files from Google Drive, standardizes column names, extracts game metadata from filenames, engineers early structural features (series counters, explosive plays, drive_down), and outputs a single combined dataframe `all_df` used by all downstream notebooks.

**Input:** Raw `.xlsx` game files from `TU_Games/` folder on Google Drive

**Output:** `all_df` — combined, cleaned play-by-play dataframe

---

### ⚠️ Path Note
When you move to your personal Google Drive after graduation, update the path in the *Find Excel Files* cell:
```
# School Drive (current)
'/content/drive/MyDrive/TUFB_EPA/TU_Games/*.xlsx'

# Personal Drive (after graduation) — update folder name/path as needed
'/content/drive/MyDrive/TUFB_EPA/TU_Games/*.xlsx'
```
The path may be the same if you copy the folder structure to your personal Drive.

## 1. Imports

In [ ]:
import os
import re
import glob
import pandas as pd
import numpy as np
from datetime import datetime

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Find Excel Files

> **To switch to personal Drive after graduation:** Update the path below to point to your personal Drive folder.

In [ ]:
# ── Find Excel files ──────────────────────────────────────────────────────────
# School Drive (current)
xlsx_files = sorted(glob.glob("/content/drive/MyDrive/TUFB_EPA/TU_Games/*.xlsx"))

# Personal Drive (after graduation) — uncomment and update path:
# xlsx_files = sorted(glob.glob("/content/drive/MyDrive/TUFB_EPA_Personal/TU_Games/*.xlsx"))

print(f"Found {len(xlsx_files)} game files:")
for f in xlsx_files:
    print(" ", os.path.basename(f))

## 4. Display Settings

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

## 5. Load & Process All Game Files

Loops through every `.xlsx` file and:
- Standardizes column names
- Removes special team scouting rows (`ODK == 'S'`)
- Fixes missing ODK on timeout rows
- Extracts game metadata (opponent, home/away, win/loss, date, score) from filename
- Flags explosive plays
- Builds offensive/defensive series counters
- Creates `drive_down` column

In [ ]:
# Explosive play thresholds
EXPLOSIVE_RUN_THRESHOLD = 12
EXPLOSIVE_PASS_THRESHOLD = 21

def is_explosive(row):
    try:
        gain = int(row["gn_ls"])
    except (ValueError, TypeError):
        return 0
    if row["play_type"] == "Run" and gain >= EXPLOSIVE_RUN_THRESHOLD:
        return 1
    if row["play_type"] == "Pass" and gain >= EXPLOSIVE_PASS_THRESHOLD:
        return 1
    return 0

all_dfs = []

for filenum, path in enumerate(xlsx_files):
    game_data = pd.read_excel(path)

    # Standardize column names
    game_data.columns = (
        game_data.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9]+", "_", regex=True)
        .str.strip("_")
    )

    # Remove scouting rows and reset index
    game_data = game_data[game_data["odk"] != "S"].reset_index(drop=True)
    game_data = game_data.replace(["nan", "NA", "", "None"], pd.NA)

    # Fix missing ODK on Timeout rows
    for i in range(1, len(game_data)):
        if (
            pd.notna(game_data.at[i, "result"])
            and game_data.at[i, "result"] == "Timeout"
            and pd.isna(game_data.at[i, "odk"])
        ):
            game_data.at[i, "odk"] = game_data.at[i - 1, "odk"]

    # ── Extract metadata from filename ───────────────────────────────────────
    filename = os.path.basename(path)

    team_name = re.match(r'^([^_]+)', filename)
    team_name = team_name.group(1) if team_name else None

    opp_name = re.match(r'^[^_]+_([^_]+)', filename)
    opp_name = opp_name.group(1) if opp_name else None

    home_away = re.search(r'\((?:W|L)_([HA])\)', filename)
    home_away = home_away.group(1) if home_away else None

    win_match = re.search(r'\((W|L)_[HA]\)', filename)
    win_flag = 1 if win_match and win_match.group(1) == "W" else 0

    date_match = re.search(r'\d{4}-\d{2}-\d{2}', filename)
    game_date = datetime.strptime(date_match.group(), "%Y-%m-%d").strftime("%m/%d/%Y") if date_match else None

    score_match = re.search(r'\)\d+-\d+_', filename)
    if score_match:
        score_clean = re.sub(r"[)_]", "", score_match.group())
        team_score, opp_score = map(int, score_clean.split("-"))
    else:
        team_score = opp_score = pd.NA

    conditions = [game_data["odk"] == "O", game_data["odk"] == "D"]
    game_data["team_name"] = np.select(conditions, [team_name, opp_name], default=team_name)
    game_data["opp_name"]  = np.select(conditions, [opp_name, team_name], default=opp_name)
    game_data["home_away"] = home_away
    game_data["win"]       = win_flag
    game_data["game_date"] = game_date
    game_data["team_pts"]  = team_score
    game_data["opp_pts"]   = opp_score

    # Convert numeric columns
    for col in ["dn", "ydline", "to_go", "gn_ls", "team_pts", "opp_pts"]:
        if col in game_data.columns:
            game_data[col] = pd.to_numeric(game_data[col], errors="coerce")

    # Explosive plays
    game_data["explosive"] = game_data.apply(is_explosive, axis=1)

    # ── Series counters ───────────────────────────────────────────────────────
    off_series_col = [0] * len(game_data)
    def_series_col = [0] * len(game_data)
    off_counter = def_counter = 0
    current_type = ""

    for i, val in enumerate(game_data["odk"]):
        if pd.notna(val) and val == "O":
            if current_type != "O":
                off_counter += 1
                current_type = "O"
            off_series_col[i] = off_counter
        elif pd.notna(val) and val == "D":
            if current_type != "D":
                def_counter += 1
                current_type = "D"
            def_series_col[i] = def_counter
        else:
            current_type = ""

    game_data["off_series"] = off_series_col
    game_data["def_series"] = def_series_col

    # Single game-level series counter
    series_col = [0] * len(game_data)
    series_counter = 0
    current_type = ""

    for i, val in enumerate(game_data["odk"]):
        if pd.notna(val) and val in ["O", "D"]:
            if val != current_type:
                series_counter += 1
                current_type = val
            series_col[i] = series_counter
        else:
            current_type = ""

    game_data["series"] = series_col

    # drive_down: first play of each drive flagged as 23
    game_data["drive_down"] = game_data["dn"]

    for series_val in game_data.loc[game_data["off_series"] != 0, "off_series"].unique():
        idx = game_data.index[(game_data["off_series"] == series_val) & (game_data["dn"] == 1)]
        if len(idx) > 0:
            game_data.loc[idx[0], "drive_down"] = 23

    for series_val in game_data.loc[game_data["def_series"] != 0, "def_series"].unique():
        idx = game_data.index[(game_data["def_series"] == series_val) & (game_data["dn"] == 1)]
        if len(idx) > 0:
            game_data.loc[idx[0], "drive_down"] = 23

    all_dfs.append(game_data)

# Combine all games
all_df = pd.concat(all_dfs, ignore_index=True)
print(f"Total plays loaded: {len(all_df)}")
all_df.info()

## 6. Drop Unused Columns

In [ ]:
# These columns are empty in the raw Hudl export or will be recreated later
cols_to_drop = [
    'unnamed_0', 'oline_scheme', 'kick_type', 'return_name_type',
    'points', 'point_differential', 'team_score', 'opponent_score', 'title'
]
# Only drop columns that actually exist (safe for future game files)
cols_to_drop = [c for c in cols_to_drop if c in all_df.columns]
all_df.drop(columns=cols_to_drop, inplace=True)
print(f"Dropped: {cols_to_drop}")
print(f"Remaining columns: {list(all_df.columns)}")

## 7. Reshape & Clean Key Columns

In [ ]:
# ── Fix dn = 0 (should be 1st down) ─────────────────────────────────────────
print("DN distribution before fix:")
print(all_df['dn'].value_counts())
all_df.loc[all_df['dn'] == 0, 'dn'] = 1
print("\nDN distribution after fix:")
print(all_df['dn'].value_counts())

In [ ]:
# ── Fix result = 1st DN (should be Complete or Rush) ────────────────────────
print("RESULT distribution before fix:")
print(all_df['result'].value_counts())
firstdn_mask = (all_df['result'] == '1st DN')
all_df.loc[firstdn_mask & (all_df['play_type'] == 'Run'),  'result'] = 'Rush'
all_df.loc[firstdn_mask & (all_df['play_type'] == 'Pass'), 'result'] = 'Complete'
print("\nRESULT distribution after fix:")
print(all_df['result'].value_counts())

In [ ]:
# ── Clean 2-point conversion result labels ───────────────────────────────────
two_pt_mask = all_df['play_type'].isin(['2 Pt.', '2 Pt. Defend'])
all_df.loc[two_pt_mask, 'result'] = all_df.loc[two_pt_mask, 'result'].replace({
    'Complete':     'Good',
    'Complete, TD': 'Good',
    'Incomplete':   'No Good',
    'Interception': 'No Good'
})
print("2pt result values after cleaning:")
print(all_df.loc[two_pt_mask, 'result'].unique())

In [ ]:
# ── Clean bare 'TD' result labels (assign Rush,TD or Complete,TD) ─────────────
bare_td_mask = (all_df['result'] == 'TD') & (all_df['odk'].isin(['O', 'D']))

print("Bare TD plays by type:")
print(all_df[bare_td_mask][['game_date', 'team_name', 'odk', 'play_type', 'result']].value_counts())

all_df.loc[bare_td_mask & (all_df['play_type'] == 'Run'),  'result'] = 'Rush, TD'
all_df.loc[bare_td_mask & (all_df['play_type'] == 'Pass'), 'result'] = 'Complete, TD'

## 8. Confirm Output

`all_df` is now ready to be used in `02_eda.ipynb`.

In [ ]:
print(f"Final shape: {all_df.shape}")
print(f"Games loaded: {all_df['game_date'].nunique()}")
print(f"ODK breakdown:")
print(all_df['odk'].value_counts())
all_df.head()